# 07 — BQ5: Top 3 Actionable Interventions for Student Retention

> **Notebook 07 of 7** | Learning Retention Analytics  
> Fifth business question: synthesizing BQ1–BQ4 findings into concrete, prioritized recommendations for a platform operator.

## Purpose and Scope

This notebook answers **BQ5: What are the top 3 actionable interventions for a platform operator?** — the final business question and the capstone of this project.

Notebooks 03–06 established *what happens* (BQ1: dropout timing), *what predicts it* (BQ2: early signals), *whether demographics or behavior matters more* (BQ3: behavior wins), and *how courses differ* (BQ4: course design vs retention). This notebook **transforms those findings into action**.

**What this notebook covers:**
- Sizing the three target segments identified across BQ1–BQ4
- Quantifying segment overlap (students may belong to multiple segments)
- Building three recommendation cards with: segment, intervention, expected impact, cost
- Ranking interventions in a priority matrix (impact vs. cost)
- Proposing a phased implementation roadmap

**What this notebook does NOT do:**
- No new statistical testing — all evidence comes from BQ1–BQ4
- No causal claims — impact estimates are projections, not measured effects
- No A/B test design — that is the natural next step after deploying interventions

**What came before:**
- **NB03** (BQ1): where and when students drop out — dropout curves, cliff detection
- **NB04** (BQ2): early behavioral signals that predict dropout — effect sizes, dose-response
- **NB05** (BQ3): demographics vs behavior — behavior predicts outcome 2–5× more strongly
- **NB06** (BQ4): course design vs retention — descriptive course profiles, exploratory correlations

> **Methodological transferability:** This synthesis pattern — segment sizing → intervention design → impact estimation → prioritization — is the standard "churn intervention playbook" in SaaS product analytics. The three segments (ghost users, feature non-adopters, early disengagers) map directly to subscription churn, fitness app retention, and freemium conversion contexts.

## Table of Contents

1. [Environment Setup](#1.-Environment-Setup)
2. [Segment Sizing — The Three Target Populations](#2.-Segment-Sizing-—-The-Three-Target-Populations)
3. [Segment Overlap](#3.-Segment-Overlap)
4. [Recommendation 1 — Ghost Student Activation](#4.-Recommendation-1-—-Ghost-Student-Activation)
5. [Recommendation 2 — First Assessment Checkpoint](#5.-Recommendation-2-—-First-Assessment-Checkpoint)
6. [Recommendation 3 — Week 3 Re-engagement Campaign](#6.-Recommendation-3-—-Week-3-Re-engagement-Campaign)
7. [Priority Matrix](#7.-Priority-Matrix)
8. [Implementation Roadmap](#8.-Implementation-Roadmap)
9. [Limitations and Caveats](#9.-Limitations-and-Caveats)
10. [Key Takeaways](#10.-Key-Takeaways)

---

**Prerequisites:**
- The ETL pipeline must have been run: `python -m run_pipeline`
- The DuckDB database at `data/db/oulad.duckdb` must contain all 5 analytical views

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K students, 7 courses, complete behavioral clickstream. License: CC-BY 4.0.

## 1. Environment Setup

We configure imports, visualization defaults, and reusable helper functions.

**Technical notes for readers:**
- All database queries go through `src.db.connection.execute_query()` — the project's DB abstraction layer (ADR-003).
- BQ5's primary SQL query lives in `sql/queries/q_bq5_segment_sizing.sql` and is loaded at runtime from disk. It sizes three intervention segments from `v_student_enriched` and `v_engagement_early`.
- Additional inline SQL queries compute impact estimates for each recommendation. These are specific to this notebook's synthesis narrative and not reusable as standalone queries (consistent with the inline query pattern in NB03).
- No statistical test imports — this is a synthesis notebook, all evidence comes from NB03–NB06.
- Figures are saved to `reports/figures/` at 150 DPI.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query

# --- Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
sns.set_theme(style='whitegrid', font_scale=1.1)

# Segment palette: one color per intervention segment for consistent
# identification across all figures. Colors chosen for semantic clarity:
# red = most critical (ghost), orange = medium (assessment), blue = re-engagement.
PALETTE_SEGMENT = {
    'Ghost students': '#C44E52',
    'Assessment non-submitters': '#DD8452',
    'Early disengagers': '#4C72B0',
}
SEGMENT_ORDER = list(PALETTE_SEGMENT.keys())

# Shared axis labels — defined as constants to avoid
# duplicated string literals flagged by static analysis
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_NON_COMPLETION_RATE = 'Non-completion rate (%)'
LABEL_NUM_STUDENTS = 'Number of students'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ5 query from SQL file ---
_bq5_sql_path = QUERIES_DIR / 'q_bq5_segment_sizing.sql'
BQ5_SQL = _bq5_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ5 query from: {_bq5_sql_path.name} ({len(BQ5_SQL):,} chars)')

# --- Prerequisite check ---
# BQ5 query depends on v_student_enriched (main student table) and
# v_engagement_early (early behavioral metrics for segment classification).
try:
    _check_se = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _check_ee = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _n_se = _check_se['n'].iloc[0]
    _n_ee = _check_ee['n'].iloc[0]
    if _n_se == 0 or _n_ee == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_student_enriched:  {_n_se:>12,} rows')
    print(f'  v_engagement_early:  {_n_ee:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Segment Sizing — The Three Target Populations

The BQ5 query (`q_bq5_segment_sizing.sql`) sizes three student segments defined by **observable, actionable criteria** — not demographics. Each segment represents a group where a targeted intervention could reduce non-completion:

| Segment | Definition | Rationale |
|---------|-----------|-----------|
| **Ghost students** | ≤1 active day AND <10 clicks in first 28 days | Enrolled but never meaningfully started. BQ2 (NB04) showed zero early engagement is the strongest dropout predictor. |
| **Assessment non-submitters** | No assessment submitted in first 28 days | Missing the first deadline is a powerful binary signal. BQ2 identified `submitted_first_assessment` as a key predictor. |
| **Early disengagers** | Had VLE activity in days 0–14 but zero activity in days 15–28 | Started but lost momentum. BQ1 (NB03) showed mid-course dropout cliffs, often at assessment points. |

**Design principle:** All three segments are defined by *behavior*, not demographics. This is consistent with BQ3's finding (NB05) that behavioral signals predict outcome 2–5× more strongly than demographic features. Interventions targeting behavior are both more effective and more ethical.

In [ ]:
# --- Load BQ5 segment sizing data ---
df_bq5 = execute_query(BQ5_SQL)

total_students = int(df_bq5['total_students'].iloc[0])

# Build a structured DataFrame for the three segments
# so we can plot and reference them consistently across sections
segments = pd.DataFrame({
    'segment': SEGMENT_ORDER,
    'n': [
        int(df_bq5['n_ghost'].iloc[0]),
        int(df_bq5['n_non_submitter'].iloc[0]),
        int(df_bq5['n_early_disengager'].iloc[0]),
    ],
    'pct_of_total': [
        float(df_bq5['pct_ghost'].iloc[0]),
        float(df_bq5['pct_non_submitter'].iloc[0]),
        float(df_bq5['pct_early_disengager'].iloc[0]),
    ],
    'non_completion_rate': [
        float(df_bq5['ghost_non_completion_rate_pct'].iloc[0]),
        float(df_bq5['non_submitter_non_completion_rate_pct'].iloc[0]),
        float(df_bq5['disengager_non_completion_rate_pct'].iloc[0]),
    ],
})

# Overall non-completion rate for context (baseline)
overall_non_completion = execute_query('''
    SELECT ROUND(100.0 * SUM(CASE WHEN completed = 0 THEN 1 ELSE 0 END)
                 / COUNT(*), 1) AS rate
    FROM v_student_enriched
''')['rate'].iloc[0]

print(f'BQ5 segment sizing: {total_students:,} total enrollments')
print(f'Overall non-completion rate: {overall_non_completion}%')
print()
print('=== Segment Summary ===')
print(segments.to_string(index=False))
print()
# How much worse each segment is compared to the overall rate
for _, row in segments.iterrows():
    excess = row['non_completion_rate'] - overall_non_completion
    print(f"  {row['segment']}: {row['non_completion_rate']:.1f}% non-completion "
          f"(+{excess:.1f} pp vs overall)")

In [ ]:
# --- Figure: Segment sizing overview ---
# 1x2 panel: segment sizes (left) and non-completion rates (right).
# Segments ordered by severity (SEGMENT_ORDER) for consistent reading.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

y_pos = np.arange(len(segments))
colors = [PALETTE_SEGMENT[s] for s in segments['segment']]

# --- Left: segment size ---
ax1.barh(y_pos, segments['n'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(segments.iterrows()):
    ax1.text(
        row['n'] + total_students * 0.01, i,
        f"{row['n']:,}  ({row['pct_of_total']:.1f}%)",
        va='center', fontsize=10, color='#333333',
    )
ax1.set_yticks(y_pos)
ax1.set_yticklabels(segments['segment'])
ax1.set_xlabel(LABEL_NUM_STUDENTS)
ax1.set_title('Segment Size')
sns.despine(ax=ax1)

# --- Right: non-completion rate ---
ax2.barh(y_pos, segments['non_completion_rate'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(segments.iterrows()):
    ax2.text(
        row['non_completion_rate'] + 1, i,
        f"{row['non_completion_rate']:.1f}%",
        va='center', fontsize=10, color='#333333',
    )
# Overall baseline reference line
ax2.axvline(
    x=overall_non_completion, color='gray', linestyle='--', linewidth=1,
    label=f'Overall: {overall_non_completion:.1f}%',
)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(segments['segment'])
ax2.set_xlabel(LABEL_NON_COMPLETION_RATE)
ax2.set_title('Non-completion Rate by Segment')
ax2.set_xlim(0, 105)
ax2.legend(loc='lower right')
sns.despine(ax=ax2)

fig.suptitle(
    'BQ5 Segment Sizing — Who Should We Target?\n'
    f'(total enrollments: {total_students:,})',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '07_segment_sizing_overview')
plt.show()

> **Segment profile:** All three segments show non-completion rates substantially above the overall baseline. The gap between each segment's rate and the platform average quantifies the "excess non-completion" — the portion potentially addressable through targeted intervention.
>
> **Note:** These segments are not mutually exclusive. A ghost student who never accessed the VLE almost certainly did not submit an assessment either. The next section quantifies this overlap to avoid double-counting when estimating aggregate impact.

## 3. Segment Overlap

Students can belong to multiple segments simultaneously. Understanding the overlap is critical for two reasons:
1. **Impact estimation:** If most ghost students are also non-submitters, the two interventions target largely the same people — their impact should not be summed naively.
2. **Intervention sequencing:** A student in multiple segments would receive multiple interventions; we should prioritize which one they get first.

The query below reproduces the BQ5 segment classification at the student level (instead of aggregating to a single row) and counts pairwise and three-way intersections.

In [ ]:
# --- Segment overlap analysis ---
# Inline query: reproduce the BQ5 segment flags at the student level
# and count pairwise/three-way intersections.
# This mirrors the CTE logic in q_bq5_segment_sizing.sql but selects
# per-student flags instead of aggregating to a single row.
OVERLAP_SQL = '''
WITH early_activity AS (
    SELECT id_student, code_module, code_presentation,
           SUM(sum_click) AS total_clicks_0_14
    FROM studentVle
    WHERE date BETWEEN 0 AND 14
    GROUP BY id_student, code_module, code_presentation
),
late_activity AS (
    SELECT id_student, code_module, code_presentation,
           SUM(sum_click) AS total_clicks_15_28
    FROM studentVle
    WHERE date BETWEEN 15 AND 28
    GROUP BY id_student, code_module, code_presentation
),
early_assessments AS (
    SELECT DISTINCT sa.id_student, a.code_module, a.code_presentation
    FROM studentAssessment sa
    JOIN assessments a ON sa.id_assessment = a.id_assessment
    WHERE a.date <= 28
),
student_segments AS (
    SELECT
        se.id_student, se.code_module, se.code_presentation,
        CASE WHEN COALESCE(ee.active_days_first_28, 0) <= 1
                  AND COALESCE(ee.total_clicks_first_28, 0) < 10
             THEN 1 ELSE 0 END AS is_ghost,
        CASE WHEN ea.id_student IS NULL
             THEN 1 ELSE 0 END AS is_non_submitter,
        CASE WHEN early_act.total_clicks_0_14 IS NOT NULL
                  AND late_act.total_clicks_15_28 IS NULL
             THEN 1 ELSE 0 END AS is_early_disengager
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    LEFT JOIN early_assessments ea
        ON se.id_student = ea.id_student
        AND se.code_module = ea.code_module
        AND se.code_presentation = ea.code_presentation
    LEFT JOIN early_activity early_act
        ON se.id_student = early_act.id_student
        AND se.code_module = early_act.code_module
        AND se.code_presentation = early_act.code_presentation
    LEFT JOIN late_activity late_act
        ON se.id_student = late_act.id_student
        AND se.code_module = late_act.code_module
        AND se.code_presentation = late_act.code_presentation
)
SELECT
    COUNT(*) AS total,
    SUM(is_ghost) AS n_ghost,
    SUM(is_non_submitter) AS n_non_sub,
    SUM(is_early_disengager) AS n_disengager,
    -- Pairwise overlaps
    SUM(CASE WHEN is_ghost = 1 AND is_non_submitter = 1
             THEN 1 ELSE 0 END) AS ghost_and_nonsub,
    SUM(CASE WHEN is_ghost = 1 AND is_early_disengager = 1
             THEN 1 ELSE 0 END) AS ghost_and_disengager,
    SUM(CASE WHEN is_non_submitter = 1 AND is_early_disengager = 1
             THEN 1 ELSE 0 END) AS nonsub_and_disengager,
    -- Three-way overlap
    SUM(CASE WHEN is_ghost = 1 AND is_non_submitter = 1 AND is_early_disengager = 1
             THEN 1 ELSE 0 END) AS all_three,
    -- Exclusive membership (belongs to exactly this one segment)
    SUM(CASE WHEN is_ghost = 1 AND is_non_submitter = 0 AND is_early_disengager = 0
             THEN 1 ELSE 0 END) AS ghost_only,
    SUM(CASE WHEN is_ghost = 0 AND is_non_submitter = 1 AND is_early_disengager = 0
             THEN 1 ELSE 0 END) AS nonsub_only,
    SUM(CASE WHEN is_ghost = 0 AND is_non_submitter = 0 AND is_early_disengager = 1
             THEN 1 ELSE 0 END) AS disengager_only,
    -- Union: in at least one segment
    SUM(CASE WHEN is_ghost = 1 OR is_non_submitter = 1 OR is_early_disengager = 1
             THEN 1 ELSE 0 END) AS in_any_segment
FROM student_segments
'''

df_overlap = execute_query(OVERLAP_SQL)
row = df_overlap.iloc[0]

# --- Print overlap summary ---
print('=== Segment Overlap Analysis ===')
print(f"Total enrollments:       {int(row['total']):>8,}")
print(f"In at least one segment: {int(row['in_any_segment']):>8,} "
      f"({100.0 * row['in_any_segment'] / row['total']:.1f}%)")
print()
print('Pairwise overlaps:')
print(f"  Ghost ∩ Non-submitter:     {int(row['ghost_and_nonsub']):>6,}")
print(f"  Ghost ∩ Disengager:        {int(row['ghost_and_disengager']):>6,}")
print(f"  Non-submitter ∩ Disengager:{int(row['nonsub_and_disengager']):>6,}")
print(f"  All three:                 {int(row['all_three']):>6,}")
print()
print('Exclusive membership (this segment only):')
print(f"  Ghost only:                {int(row['ghost_only']):>6,}")
print(f"  Non-submitter only:        {int(row['nonsub_only']):>6,}")
print(f"  Disengager only:           {int(row['disengager_only']):>6,}")

In [ ]:
# --- Figure: Segment overlap ---
# Horizontal bar showing exclusive and overlapping membership counts.
# This helps understand how much the interventions would target the same students.
overlap_data = pd.DataFrame({
    'category': [
        'Ghost only',
        'Non-submitter only',
        'Disengager only',
        'Ghost + Non-submitter',
        'Non-submitter + Disengager',
        'Ghost + Disengager',
        'All three',
    ],
    'n': [
        int(row['ghost_only']),
        int(row['nonsub_only']),
        int(row['disengager_only']),
        # Pairwise exclusive: subtract three-way overlap to avoid double-counting
        int(row['ghost_and_nonsub']) - int(row['all_three']),
        int(row['nonsub_and_disengager']) - int(row['all_three']),
        int(row['ghost_and_disengager']) - int(row['all_three']),
        int(row['all_three']),
    ],
})

# Filter out zero-count categories for cleaner visualization
overlap_data = overlap_data[overlap_data['n'] > 0].sort_values('n', ascending=True)

# Color mapping: exclusive categories get the segment color,
# overlap categories get a neutral gray
overlap_colors = []
for cat in overlap_data['category']:
    if cat == 'Ghost only':
        overlap_colors.append(PALETTE_SEGMENT['Ghost students'])
    elif cat == 'Non-submitter only':
        overlap_colors.append(PALETTE_SEGMENT['Assessment non-submitters'])
    elif cat == 'Disengager only':
        overlap_colors.append(PALETTE_SEGMENT['Early disengagers'])
    else:
        overlap_colors.append('#999999')

fig, ax = plt.subplots(figsize=FIG_SIZE)
y_pos = np.arange(len(overlap_data))

ax.barh(y_pos, overlap_data['n'].values, color=overlap_colors, edgecolor='white')
for i, (_, cat_row) in enumerate(overlap_data.iterrows()):
    n_val = cat_row['n']
    pct = 100.0 * n_val / total_students
    ax.text(
        n_val + total_students * 0.005, i,
        f'{n_val:,}  ({pct:.1f}%)',
        va='center', fontsize=9, color='#333333',
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(overlap_data['category'].values)
ax.set_xlabel(LABEL_NUM_STUDENTS)
ax.set_title(
    'Segment Overlap — Exclusive and Shared Membership\n'
    '(gray bars = students in multiple segments)'
)
sns.despine()
fig.tight_layout()
save_fig(fig, '07_segment_overlap')
plt.show()

# --- Key overlap metric for downstream impact estimation ---
# Percentage of ghost students who are also non-submitters
ghost_total = int(row['n_ghost'])
ghost_nonsub_overlap = int(row['ghost_and_nonsub'])
if ghost_total > 0:
    overlap_pct = 100.0 * ghost_nonsub_overlap / ghost_total
    print(f'\nGhost ∩ Non-submitter overlap: {ghost_nonsub_overlap:,} of '
          f'{ghost_total:,} ghost students ({overlap_pct:.0f}%)')
    print('  → Ghost activation and assessment reminders would largely target '
          'the same students.')

> **Overlap insight:** Ghost students and assessment non-submitters overlap heavily — a student who never accesses the VLE cannot submit an assessment. This means **Recommendations 1 and 2 largely target the same population** from different angles. Early disengagers, by definition, had *some* initial activity, so they overlap less with ghost students. This makes Recommendation 3 an independent intervention targeting a different failure mode.
>
> **Implication for impact estimation:** Because of the ghost–non-submitter overlap, the combined impact of all three interventions is **less than the sum of individual impacts**. Each recommendation section below estimates impact independently; the Priority Matrix (Section 7) adjusts for overlap.

## 4. Recommendation 1 — Ghost Student Activation

### The Problem

Ghost students enroll but never meaningfully engage with the course. They represent the most severe form of non-completion: not a failure to persist, but a failure to *start*.

### Evidence from BQ1–BQ4

- **BQ2 (NB04):** Early engagement (active days, total clicks in first 28 days) has the largest effect size among all behavioral predictors. Students with zero early activity have near-zero completion rates.
- **BQ3 (NB05):** Behavior predicts outcome 2–5× more strongly than demographics. This means we should target *what students do* (or fail to do), not *who they are*.
- **BQ1 (NB03):** A significant fraction of withdrawals happen in the first two weeks — before most students have even established a study routine.

### Proposed Intervention

| Element | Details |
|---------|---------|
| **Trigger** | Student has zero VLE activity by day 3 of the course |
| **Action** | Automated activation sequence: day-3 welcome email with "first step" link to a single easy resource, day-7 follow-up if still inactive |
| **Channel** | Email + in-platform notification |
| **Cost** | **Low** — email automation only, no platform changes required |

### Impact Estimation

In [ ]:
# --- Impact estimation: Ghost student activation ---
# Compare completion rates between ghost students and active students
# to quantify the gap that activation could partially close.
df_ghost_impact = execute_query('''
    SELECT
        CASE
            WHEN COALESCE(ee.active_days_first_28, 0) <= 1
                 AND COALESCE(ee.total_clicks_first_28, 0) < 10
            THEN 'Ghost'
            ELSE 'Active'
        END AS student_type,
        COUNT(*) AS n,
        SUM(se.completed) AS n_completed,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    GROUP BY 1
''')

ghost_row = df_ghost_impact[df_ghost_impact['student_type'] == 'Ghost'].iloc[0]
active_row = df_ghost_impact[df_ghost_impact['student_type'] == 'Active'].iloc[0]

ghost_n = int(ghost_row['n'])
ghost_rate = float(ghost_row['completion_rate_pct'])
active_rate = float(active_row['completion_rate_pct'])

print('=== Ghost vs Active Completion Rates ===')
print(df_ghost_impact.to_string(index=False))
print()

# Scenario analysis: if activation emails convert X% of ghost students
# to minimal engagement, and those converted students achieve the
# platform-average completion rate (conservative estimate)

print('=== Scenario Analysis ===')
print(f'Ghost students: {ghost_n:,} with {ghost_rate:.1f}% completion rate')
print(f'Active students: {active_rate:.1f}% completion rate')
print(f'Platform average: {100.0 - overall_non_completion:.1f}% completion rate')
print()

# Store scenario results for the priority matrix
ghost_scenarios = {}
for conversion_pct in [10, 20, 30]:
    converted = int(ghost_n * conversion_pct / 100)
    # Conservative: converted students achieve the overall average, not the active rate
    target_rate = (100.0 - overall_non_completion) / 100.0
    current_rate = ghost_rate / 100.0
    additional_completions = int(converted * (target_rate - current_rate))
    ghost_scenarios[conversion_pct] = additional_completions
    print(f'  If {conversion_pct}% of ghosts activate → ~{additional_completions:,} '
          f'additional completions ({converted:,} converted)')

# Save the middle scenario for the priority matrix
rec1_impact = ghost_scenarios[20]

> **Interpretation:** The completion rate gap between ghost and active students is substantial. Even a modest activation rate (20%) would produce meaningful additional completions because the segment is large and the gap is wide.
>
> **Assumption transparency:** The scenario assumes converted students achieve the *platform average* completion rate — a conservative estimate. In practice, late-activating students may perform worse than those who started on time. This would reduce the estimated impact.

## 5. Recommendation 2 — First Assessment Checkpoint

### The Problem

Students who miss the first assessment deadline are signaling disengagement. Missing this early milestone breaks the feedback loop that keeps students connected to the course — they lose the sense of progress that early assessment provides.

### Evidence from BQ1–BQ4

- **BQ2 (NB04):** `submitted_first_assessment` is a powerful binary predictor of completion. The effect size is among the largest of all early signals.
- **BQ4 (NB06):** Courses with higher assessment density (more frequent checkpoints) show suggestive patterns with retention. Regular assessment may provide structure that keeps students on track.
- **BQ1 (NB03):** Dropout cliffs often coincide with assessment deadlines, suggesting that assessments are decision points where students either commit or leave.

### Proposed Intervention

| Element | Details |
|---------|---------|
| **Trigger** | First assessment deadline is 3 days away and student has not submitted |
| **Action** | Automated reminder with assessment preview: "Here's what to expect — the first assessment covers X and takes approximately Y minutes" |
| **Channel** | Email + in-platform notification + optional SMS |
| **Cost** | **Medium** — requires deadline-aware notification system integrated with course calendar |

### Impact Estimation

In [ ]:
# --- Impact estimation: First assessment checkpoint ---
# Compare completion rates between students who submitted the first
# assessment (due within 28 days) and those who did not.
df_assess_impact = execute_query('''
    SELECT
        CASE
            WHEN ea.id_student IS NOT NULL THEN 'Submitted'
            ELSE 'Not submitted'
        END AS assessment_status,
        COUNT(*) AS n,
        SUM(se.completed) AS n_completed,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_student_enriched se
    LEFT JOIN (
        SELECT DISTINCT sa.id_student, a.code_module, a.code_presentation
        FROM studentAssessment sa
        JOIN assessments a ON sa.id_assessment = a.id_assessment
        WHERE a.date <= 28
    ) ea
        ON se.id_student = ea.id_student
        AND se.code_module = ea.code_module
        AND se.code_presentation = ea.code_presentation
    GROUP BY 1
''')

sub_row = df_assess_impact[df_assess_impact['assessment_status'] == 'Submitted'].iloc[0]
nosub_row = df_assess_impact[df_assess_impact['assessment_status'] == 'Not submitted'].iloc[0]

nosub_n = int(nosub_row['n'])
nosub_rate = float(nosub_row['completion_rate_pct'])
sub_rate = float(sub_row['completion_rate_pct'])

print('=== Submitter vs Non-submitter Completion Rates ===')
print(df_assess_impact.to_string(index=False))
print()

# Scenario analysis: if reminders convert X% of non-submitters to submitters
print('=== Scenario Analysis ===')
print(f'Non-submitters: {nosub_n:,} with {nosub_rate:.1f}% completion rate')
print(f'Submitters: {sub_rate:.1f}% completion rate')
print()

assess_scenarios = {}
for conversion_pct in [10, 15, 25]:
    converted = int(nosub_n * conversion_pct / 100)
    # Converted students achieve submitter rate (they actually submitted)
    additional_completions = int(converted * (sub_rate - nosub_rate) / 100.0)
    assess_scenarios[conversion_pct] = additional_completions
    print(f'  If {conversion_pct}% of non-submitters submit → ~{additional_completions:,} '
          f'additional completions ({converted:,} converted)')

# Save the middle scenario for the priority matrix
rec2_impact = assess_scenarios[15]

> **Interpretation:** The gap between submitters and non-submitters is stark. Assessment submission is both a *signal* (it reveals commitment) and a *mechanism* (it creates accountability). This dual nature makes it an ideal intervention point.
>
> **Caution on causality:** Submitting the assessment may not *cause* completion — both may be driven by an underlying motivation factor. The intervention works only if the reminder nudges students who *would have submitted* with a small push, not if it forces reluctant students through a gate. This is why the conversion rate assumptions are conservative (10–25%).

## 6. Recommendation 3 — Week 3 Re-engagement Campaign

### The Problem

Early disengagers are students who *started* the course — they had VLE activity in the first two weeks — but then stopped entirely in weeks 3–4. Unlike ghost students who never began, these students demonstrated initial interest but lost momentum.

### Evidence from BQ1–BQ4

- **BQ1 (NB03):** Dropout curves show mid-course cliffs that often align with assessment deadlines or the transition from introductory to core content. Weeks 3–4 are a critical inflection point.
- **BQ2 (NB04):** `last_active_day_in_window` is a significant predictor — students whose last activity is early in the window are unlikely to complete.
- **BQ3 (NB05):** The behavioral signal (activity drop) is more predictive than any demographic factor. The intervention should target the behavior, not the student's profile.

### Proposed Intervention

| Element | Details |
|---------|---------|
| **Trigger** | Student had ≥1 active day in days 0–14 but zero activity in days 15–17 (3-day inactivity after initial engagement) |
| **Action** | "We miss you" email at day 18: personalized progress summary ("You completed X% of week 1–2 activities"), social proof ("Students like you who re-engaged at this point completed the course Y% of the time"), and a direct link to the next resource |
| **Channel** | Email + in-platform notification |
| **Cost** | **Medium-High** — requires real-time activity tracking pipeline and personalized message generation |

### Impact Estimation

In [ ]:
# --- Impact estimation: Week 3 re-engagement ---
# Compare completion rates between students who sustained activity through
# weeks 3-4 and those who disengaged (had activity in days 0-14 only).
# We restrict to students who had at least SOME initial activity (days 0-14),
# since ghost students are addressed by Recommendation 1.
df_disengage_impact = execute_query('''
    WITH initially_active AS (
        SELECT DISTINCT id_student, code_module, code_presentation
        FROM studentVle
        WHERE date BETWEEN 0 AND 14
    ),
    sustained AS (
        SELECT DISTINCT id_student, code_module, code_presentation
        FROM studentVle
        WHERE date BETWEEN 15 AND 28
    )
    SELECT
        CASE
            WHEN sus.id_student IS NULL THEN 'Disengaged (weeks 3-4)'
            ELSE 'Sustained activity'
        END AS engagement_pattern,
        COUNT(*) AS n,
        SUM(se.completed) AS n_completed,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_student_enriched se
    JOIN initially_active ia
        ON se.id_student = ia.id_student
        AND se.code_module = ia.code_module
        AND se.code_presentation = ia.code_presentation
    LEFT JOIN sustained sus
        ON se.id_student = sus.id_student
        AND se.code_module = sus.code_module
        AND se.code_presentation = sus.code_presentation
    GROUP BY 1
''')

dis_row = df_disengage_impact[
    df_disengage_impact['engagement_pattern'] == 'Disengaged (weeks 3-4)'
].iloc[0]
sus_row = df_disengage_impact[
    df_disengage_impact['engagement_pattern'] == 'Sustained activity'
].iloc[0]

dis_n = int(dis_row['n'])
dis_rate = float(dis_row['completion_rate_pct'])
sus_rate = float(sus_row['completion_rate_pct'])

print('=== Sustained vs Disengaged Completion Rates ===')
print('(restricted to students with initial activity in days 0-14)')
print(df_disengage_impact.to_string(index=False))
print()

# Scenario analysis
print('=== Scenario Analysis ===')
print(f'Early disengagers: {dis_n:,} with {dis_rate:.1f}% completion rate')
print(f'Sustained students: {sus_rate:.1f}% completion rate')
print()

disengage_scenarios = {}
for conversion_pct in [10, 15, 25]:
    converted = int(dis_n * conversion_pct / 100)
    # Re-engaged students achieve a rate between disengaged and sustained
    # (conservative: halfway, not the full sustained rate)
    target_rate = (dis_rate + sus_rate) / 2.0 / 100.0
    current_rate = dis_rate / 100.0
    additional_completions = int(converted * (target_rate - current_rate))
    disengage_scenarios[conversion_pct] = additional_completions
    print(f'  If {conversion_pct}% re-engage → ~{additional_completions:,} '
          f'additional completions ({converted:,} re-engaged)')

# Save the middle scenario for the priority matrix
rec3_impact = disengage_scenarios[15]

> **Interpretation:** Early disengagers have already demonstrated willingness to engage — they are not ghost students. This means a re-engagement nudge has a plausible mechanism: reminding someone who *was* active to come back. The impact estimate uses a conservative target (halfway between disengaged and sustained rates) because re-engagement after a gap is harder than sustained momentum.
>
> **Why this is the most complex intervention:** Unlike simple email automation (Rec 1) or deadline-aware reminders (Rec 2), this intervention requires tracking individual activity patterns in near-real-time and generating personalized messages. The higher implementation cost is justified only if the segment is large enough — which the BQ5 sizing confirms.

## 7. Priority Matrix

The priority matrix ranks the three interventions by **estimated impact** (additional completions at the middle scenario) vs. **implementation cost** (Low / Medium / Medium-High). This framework helps a platform operator decide *what to build first*.

Cost is scored on a 1–3 scale:
- **1 (Low):** Email automation only, no platform integration
- **2 (Medium):** Requires deadline-aware triggers or course calendar integration
- **3 (Medium-High):** Requires real-time activity tracking and personalized messaging

In [ ]:
# --- Priority matrix data ---
priority = pd.DataFrame({
    'rank': [1, 2, 3],
    'recommendation': [
        'Ghost Student Activation',
        'First Assessment Checkpoint',
        'Week 3 Re-engagement',
    ],
    'segment': SEGMENT_ORDER,
    'segment_size': [
        segments.loc[0, 'n'],
        segments.loc[1, 'n'],
        segments.loc[2, 'n'],
    ],
    'non_completion_rate': [
        segments.loc[0, 'non_completion_rate'],
        segments.loc[1, 'non_completion_rate'],
        segments.loc[2, 'non_completion_rate'],
    ],
    'est_additional_completions': [rec1_impact, rec2_impact, rec3_impact],
    'cost_score': [1, 2, 3],
    'cost_label': ['Low', 'Medium', 'Medium-High'],
})

print('=== Priority Matrix (ranked by impact/cost ratio) ===')
print()
print(priority[[
    'rank', 'recommendation', 'segment_size',
    'non_completion_rate', 'est_additional_completions', 'cost_label',
]].to_string(index=False))

print()
total_impact = priority['est_additional_completions'].sum()
print(f'Total estimated additional completions (all 3 interventions): ~{total_impact:,}')
print('''Note: actual total will be lower due to segment overlap
      (ghost ∩ non-submitter).''')

In [ ]:
# --- Figure: Priority matrix ---
# Bubble chart: x = cost (categorical), y = estimated impact,
# bubble size proportional to segment size, colored by segment.
fig, ax = plt.subplots(figsize=FIG_SIZE)

# Scale bubble sizes for visual clarity (200-800 pixel range)
sizes = priority['segment_size'].values.astype(float)
size_min, size_max = sizes.min(), sizes.max()
# Handle edge case where all segments are the same size
if size_max > size_min:
    bubble_sizes = 200 + 600 * (sizes - size_min) / (size_max - size_min)
else:
    bubble_sizes = np.full_like(sizes, 400.0)

colors = [PALETTE_SEGMENT[s] for s in priority['segment']]

# Add subtle quadrant shading to guide interpretation
ax.axhspan(
    ymin=0, ymax=ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 100,
    xmin=0, xmax=0.33, alpha=0.04, color='green',
)

for i, (_, p_row) in enumerate(priority.iterrows()):
    ax.scatter(
        p_row['cost_score'], p_row['est_additional_completions'],
        s=bubble_sizes[i], color=colors[i],
        edgecolor='white', linewidth=2, zorder=3, alpha=0.85,
    )
    # Label each bubble with recommendation name
    ax.annotate(
        p_row['recommendation'],
        (p_row['cost_score'], p_row['est_additional_completions']),
        fontsize=9, fontweight='bold', ha='center', va='bottom',
        xytext=(0, 12), textcoords='offset points',
    )
    # Annotate with impact number inside/below bubble
    ax.annotate(
        f"~{p_row['est_additional_completions']:,}",
        (p_row['cost_score'], p_row['est_additional_completions']),
        fontsize=8, ha='center', va='top',
        xytext=(0, -10), textcoords='offset points', color='#555555',
    )

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Low', 'Medium', 'Medium-High'])
ax.set_xlabel('Implementation Cost')
ax.set_ylabel('Estimated Additional Completions\n(middle scenario)')
ax.set_title(
    'Priority Matrix — Impact vs Cost\n'
    '(bubble size = segment size)'
)
ax.set_xlim(0.4, 3.6)
# Ensure y-axis starts at 0 for honest visual comparison
ax.set_ylim(bottom=0)
sns.despine()
fig.tight_layout()
save_fig(fig, '07_priority_matrix')
plt.show()

> **Reading the matrix:** The ideal intervention sits in the top-left corner — high impact, low cost. Ghost Student Activation is the clear "quick win": it targets the largest segment with the highest non-completion rate, and requires only email automation. The First Assessment Checkpoint offers solid impact at moderate cost. The Week 3 Re-engagement Campaign has the highest implementation complexity but targets a distinct population (not overlapping with ghosts), making it a valuable addition.
>
> **Priority ranking:** (1) Ghost Activation — start here, (2) Assessment Checkpoint — build next, (3) Re-engagement Campaign — invest when infrastructure is ready.

## 8. Implementation Roadmap

A phased rollout minimizes risk and allows each intervention to be validated before investing in the next.

### Phase 1: Weeks 1–2 — Ghost Student Activation (Quick Win)

| Step | Action | Owner |
|------|--------|-------|
| 1.1 | Define email template: welcome message + "first step" resource link | Content team |
| 1.2 | Configure day-3 trigger: student has zero VLE activity since enrollment | Engineering |
| 1.3 | Configure day-7 follow-up: still inactive after first email | Engineering |
| 1.4 | A/B test: 50% receive emails, 50% control | Data team |
| 1.5 | Measure: activation rate (% of ghosts who access VLE within 14 days) | Data team |

**Success metric:** ≥15% of targeted ghost students access the VLE within 7 days of receiving the email.

### Phase 2: Weeks 3–4 — First Assessment Checkpoint

| Step | Action | Owner |
|------|--------|-------|
| 2.1 | Integrate assessment calendar with notification system | Engineering |
| 2.2 | Define reminder template: assessment preview + time estimate | Content team |
| 2.3 | Configure trigger: 3 days before first deadline, student has not submitted | Engineering |
| 2.4 | A/B test against control group | Data team |
| 2.5 | Measure: submission rate lift and downstream completion rate | Data team |

**Success metric:** ≥10% increase in first assessment submission rate among targeted students.

### Phase 3: Weeks 5–8 — Week 3 Re-engagement Campaign

| Step | Action | Owner |
|------|--------|-------|
| 3.1 | Build real-time activity tracking pipeline (daily activity flags) | Engineering |
| 3.2 | Implement personalized message generation (progress summary, peer stats) | Engineering + Content |
| 3.3 | Configure trigger: 3 consecutive inactive days after initial engagement | Engineering |
| 3.4 | A/B test with personalized vs generic re-engagement messages | Data team |
| 3.5 | Measure: re-engagement rate and completion rate lift | Data team |

**Success metric:** ≥10% of targeted disengagers return to VLE activity within 7 days.

### Cross-cutting Principle

All three interventions target **behavior, not demographics** — consistent with BQ3's finding that behavioral signals are stronger predictors. This is both more effective (targeting the actionable variable) and more ethical (avoiding demographic profiling).

## 9. Limitations and Caveats

These recommendations are evidence-based projections, not guaranteed outcomes. Several limitations must be acknowledged:

### Impact estimation assumptions

- **Conversion rates are assumed, not measured.** The 10–25% conversion scenarios are plausible based on industry benchmarks for email-based interventions in education, but they have not been validated with OULAD data (no A/B test data exists).
- **Target completion rates are conservative.** For ghost activation, we assume converted students achieve the platform average (not the active-student rate). For re-engagement, we assume halfway between disengaged and sustained rates. Actual outcomes may be higher or lower.
- **Segment overlap inflates naive impact sums.** Ghost students and assessment non-submitters overlap heavily. The combined impact of all three interventions is less than the sum of individual estimates.

### Data limitations

- **Observational data only.** All effect sizes and completion rate differences are associational, not causal. A student who submits the first assessment may complete because of motivation, not because of the submission itself.
- **Historical patterns may not hold.** OULAD covers 2013–2014 cohorts. Student behavior, platform features, and the online learning landscape have changed significantly since then.
- **No cost data.** Implementation cost estimates (Low/Medium/Medium-High) are qualitative. Actual engineering effort depends on the platform's existing infrastructure.

### Ethical considerations

- All interventions target behavior, not demographics — no student is profiled by gender, age, disability, or socioeconomic status.
- Automated interventions should include an opt-out mechanism to respect student autonomy.
- "Ghost students" may have valid reasons for not engaging (changed circumstances, enrolled in error). The intervention should inform, not pressure.

## 10. Key Takeaways

### BQ5 Answer: Top 3 Actionable Interventions

| Priority | Intervention | Segment Size | Non-completion Rate | Cost | Key Evidence |
|----------|-------------|-------------|--------------------:|------|-------------|
| 1 | **Ghost Student Activation** | See sizing above | See rate above | Low | BQ2: early engagement is strongest predictor |
| 2 | **First Assessment Checkpoint** | See sizing above | See rate above | Medium | BQ2: assessment submission is a key signal |
| 3 | **Week 3 Re-engagement** | See sizing above | See rate above | Medium-High | BQ1: mid-course dropout cliffs at weeks 3–4 |

### Synthesis Across All 5 Business Questions

| NB | BQ | Key Finding |
|----|-----|-------------|
| 03 | BQ1 | Dropout is not uniform — it concentrates in cliffs at specific course milestones. Pre-course withdrawal is a significant fraction. |
| 04 | BQ2 | Early behavioral signals (active days, clicks, assessment submission) predict dropout with large effect sizes. The first 28 days are the critical window. |
| 05 | BQ3 | Behavior predicts outcome 2–5× more strongly than demographics. Interventions should target what students *do*, not who they *are*. |
| 06 | BQ4 | Completion rates vary substantially across courses. Course design features (assessment density, resource diversity) show suggestive associations with retention. |
| 07 | BQ5 | Three actionable interventions — ghost activation, assessment checkpoint, re-engagement campaign — target the largest at-risk segments with data-estimated impact. |

### The Analytical Pipeline Is Complete

This notebook concludes the analytical work. All findings will be consolidated in `REPORT.md` for stakeholder consumption. The next step is deployment: A/B testing each intervention, measuring actual conversion rates, and iterating.

---

**Reproducibility:** All figures are saved to `reports/figures/`. To reproduce this notebook, run `python -m run_pipeline` first, then execute all cells in order.

> **From analysis to action:** This project started with a dataset and five questions. Seven notebooks later, we have a complete picture: when students leave, what predicts it, what matters more (behavior), how courses differ, and — most importantly — what a platform operator can do about it. The three interventions proposed here are not speculative: they are sized by real data, supported by statistical evidence, and ranked by feasibility.
>
> The gap between analysis and impact is an A/B test. That is the next step.